# HGTCM P3 - BART OCR Correction

This notebook remains the readable methods reference for BART correction. The canonical server rerun path now mirrors this logic through `scratch_target/scripts/run_bart_correction_experiment.py` and the source-specific server wrappers.

## Visible Code: Shared Utilities

In [ ]:
# ## Workspace for manifests/runs/reports; imports for BART and plotting.
import csv
import inspect
import json
import os
import random
import shutil
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option('display.max_colwidth', 140)


@dataclass(frozen=True)
class Workspace:
    root: Path

    @property
    def manifests_dir(self) -> Path:
        return self.root / 'manifests'

    @property
    def runs_dir(self) -> Path:
        return self.root / 'runs'

    @property
    def reports_dir(self) -> Path:
        return self.root / 'reports'

    @property
    def models_dir(self) -> Path:
        return self.root / 'models'


def workspace_from(value: str | None = None) -> Workspace:
    return Workspace(Path(value or '.').resolve())


def utc_run_id(prefix: str) -> str:
    return f"{prefix}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def read_json(path: Path) -> object:
    return json.loads(path.read_text(encoding='utf-8'))


def write_json(path: Path, payload: object) -> Path:
    ensure_dir(path.parent)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    return path


def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]


def write_jsonl(path: Path, rows: list[dict]) -> Path:
    ensure_dir(path.parent)
    with path.open('w', encoding='utf-8', newline='\n') as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + '\n')
    return path


def normalize_text(text: str) -> str:
    return ' '.join((text or '').replace('\r\n', '\n').replace('\r', '\n').split())


def levenshtein(a: list | str, b: list | str) -> int:
    if a == b:
        return 0
    if len(a) < len(b):
        a, b = b, a
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        current = [i]
        for j, cb in enumerate(b, 1):
            insert = current[j - 1] + 1
            delete = previous[j] + 1
            replace = previous[j - 1] + (ca != cb)
            current.append(min(insert, delete, replace))
        previous = current
    return previous[-1]


def cer(reference: str, hypothesis: str) -> float:
    reference = reference or ''
    hypothesis = hypothesis or ''
    if not reference:
        return 0.0 if not hypothesis else 1.0
    return levenshtein(reference, hypothesis) / len(reference)


def wer(reference: str, hypothesis: str) -> float:
    reference_tokens = (reference or '').split()
    hypothesis_tokens = (hypothesis or '').split()
    if not reference_tokens:
        return 0.0 if not hypothesis_tokens else 1.0
    return levenshtein(reference_tokens, hypothesis_tokens) / len(reference_tokens)


def env_flag(name: str, default: bool = False) -> bool:
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() in {'1', 'true', 'yes', 'on'}


def env_int_or_none(name: str, default: int | None = None) -> int | None:
    raw = os.environ.get(name)
    if raw is None or not raw.strip():
        return default
    if raw.strip().lower() in {'none', 'full', 'all'}:
        return None
    return int(raw)


def summary_table(summary: dict, title: str | None = None) -> None:
    if title:
        display(Markdown(f'### {title}'))
    display(pd.DataFrame([summary]))

def write_csv(path: Path, rows: list[dict]) -> Path:
    ensure_dir(path.parent)
    if not rows:
        path.write_text('', encoding='utf-8')
        return path
    fieldnames = list(rows[0].keys())
    with path.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return path


def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except Exception:
        return


## Configuration

In [ ]:
# ## Correction run env: which predictions file, BART base path, run frozen vs finetune flags.
WORKSPACE = workspace_from(os.environ.get('PROJECT_DECODE_ROOT', '.'))
DATA_RUN_LABEL = os.environ.get('DATA_RUN_LABEL', 'pilot').strip() or 'pilot'
CORRECTION_SOURCE_NAME = os.environ.get('CORRECTION_SOURCE_NAME', 'target_only').strip() or 'target_only'
CORRECTION_PREDICTIONS_PATH = os.environ.get('CORRECTION_PREDICTIONS_PATH', '').strip()
CORRECTION_MANIFEST_PATH = os.environ.get('CORRECTION_MANIFEST_PATH', '').strip()
BART_BASE_CHECKPOINT_OVERRIDE = os.environ.get('BART_BASE_CHECKPOINT_OVERRIDE', '').strip()
CORRECTION_PROFILE = os.environ.get('CORRECTION_PROFILE', 'smoke').strip().lower()

RUN_FROZEN_BART = env_flag('RUN_FROZEN_BART', False)
RUN_BART_FINETUNE = env_flag('RUN_BART_FINETUNE', False)

HISTORICAL_BART_MODEL = 'ancatmara/bart-ocr-correction-ga-en'
FALLBACK_BART_MODEL = 'facebook/bart-base'

USE_ACCEL_CORRECTION = CORRECTION_PROFILE in {'cloud', 'full', 'gpu', 'h100', 'h200'}
USE_H200_CORRECTION = CORRECTION_PROFILE in {'h200', 'full'}
if USE_ACCEL_CORRECTION:
    MAX_FROZEN_TEST_SAMPLES = env_int_or_none('MAX_FROZEN_TEST_SAMPLES', None)
    MAX_FINETUNE_TRAIN_SAMPLES = env_int_or_none('MAX_FINETUNE_TRAIN_SAMPLES', None)
    MAX_FINETUNE_VAL_SAMPLES = env_int_or_none('MAX_FINETUNE_VAL_SAMPLES', None)
    MAX_FINETUNE_TEST_SAMPLES = env_int_or_none('MAX_FINETUNE_TEST_SAMPLES', None)
    BART_EPOCHS = env_int_or_none('BART_EPOCHS', 6 if USE_H200_CORRECTION else 5)
else:
    MAX_FROZEN_TEST_SAMPLES = env_int_or_none('MAX_FROZEN_TEST_SAMPLES', 10)
    MAX_FINETUNE_TRAIN_SAMPLES = env_int_or_none('MAX_FINETUNE_TRAIN_SAMPLES', 64)
    MAX_FINETUNE_VAL_SAMPLES = env_int_or_none('MAX_FINETUNE_VAL_SAMPLES', 32)
    MAX_FINETUNE_TEST_SAMPLES = env_int_or_none('MAX_FINETUNE_TEST_SAMPLES', 10)
    BART_EPOCHS = env_int_or_none('BART_EPOCHS', 3)

BART_BATCH_SIZE = env_int_or_none('BART_BATCH_SIZE', 32 if USE_H200_CORRECTION else 16 if USE_ACCEL_CORRECTION else 4) or 4
BART_GRADIENT_ACCUMULATION = env_int_or_none('BART_GRADIENT_ACCUMULATION', 1 if USE_ACCEL_CORRECTION else 2) or 1
BART_LEARNING_RATE = float(os.environ.get('BART_LEARNING_RATE', '5e-5'))
BART_WARMUP_RATIO = float(os.environ.get('BART_WARMUP_RATIO', '0.1'))
BART_LR_SCHEDULER = os.environ.get('BART_LR_SCHEDULER', 'cosine')
MAX_SOURCE_LENGTH = env_int_or_none('MAX_SOURCE_LENGTH', 256) or 256
MAX_TARGET_LENGTH = env_int_or_none('MAX_TARGET_LENGTH', 256) or 256
GENERATION_NUM_BEAMS = env_int_or_none('GENERATION_NUM_BEAMS', 1) or 1
BART_DATALOADER_WORKERS = env_int_or_none('BART_DATALOADER_WORKERS', 4 if USE_ACCEL_CORRECTION else 0) or 0
BART_PRECISION = os.environ.get('BART_PRECISION', 'auto').strip().lower()
BART_SAVE_TOTAL_LIMIT = env_int_or_none('BART_SAVE_TOTAL_LIMIT', 6) or 6
BART_SEED = env_int_or_none('BART_SEED', 42) or 42
BART_DISABLE_EARLY_STOPPING = env_flag('BART_DISABLE_EARLY_STOPPING', True)
BART_EARLY_STOPPING_PATIENCE = env_int_or_none('BART_EARLY_STOPPING_PATIENCE', 2) or 2
BART_EARLY_STOPPING_THRESHOLD = float(os.environ.get('BART_EARLY_STOPPING_THRESHOLD', '0.001'))

set_all_seeds(BART_SEED)

CORRECTION_RUN_ID = utc_run_id(f'correction_bart_{DATA_RUN_LABEL}_{CORRECTION_SOURCE_NAME}')
CORRECTION_RUN_DIR = ensure_dir(WORKSPACE.reports_dir / CORRECTION_RUN_ID)

summary_table(
    {
        'workspace': str(WORKSPACE.root),
        'data_run_label': DATA_RUN_LABEL,
        'correction_source_name': CORRECTION_SOURCE_NAME,
        'correction_predictions_path_override': CORRECTION_PREDICTIONS_PATH or None,
        'correction_manifest_path_override': CORRECTION_MANIFEST_PATH or None,
        'bart_base_checkpoint_override': BART_BASE_CHECKPOINT_OVERRIDE or None,
        'correction_profile': CORRECTION_PROFILE,
        'run_frozen_bart': RUN_FROZEN_BART,
        'run_bart_finetune': RUN_BART_FINETUNE,
        'historical_bart_model': HISTORICAL_BART_MODEL,
        'fallback_bart_model': FALLBACK_BART_MODEL,
        'correction_run_id': CORRECTION_RUN_ID,
        'correction_run_dir': str(CORRECTION_RUN_DIR),
        'bart_seed': BART_SEED,
        'bart_save_total_limit': BART_SAVE_TOTAL_LIMIT,
        'bart_disable_early_stopping': BART_DISABLE_EARLY_STOPPING,
        'bart_early_stopping_patience': BART_EARLY_STOPPING_PATIENCE,
        'bart_early_stopping_threshold': BART_EARLY_STOPPING_THRESHOLD,
    },
    'Correction configuration',
)


## Build Correction Pairs From OCR Predictions

In [ ]:
# ## Discover latest OCR prediction jsonl paths and build (ocr, ref) text fields for pairs.
# Why: correction pairs isolate post-OCR language modeling effects from OCR detector effects.
def prediction_text_from_row(row: dict) -> str:
    for key in ('prediction_normalized', 'prediction', 'prediction_text', 'prediction_raw', 'text'):
        value = row.get(key)
        if value:
            return normalize_text(str(value))
    return ''


def discover_prediction_paths_for_source(workspace: Workspace, run_label: str, source_name: str) -> tuple[Path | None, dict[str, Path]]:
    if CORRECTION_PREDICTIONS_PATH:
        override_path = Path(CORRECTION_PREDICTIONS_PATH)
        if not override_path.exists():
            raise FileNotFoundError(f'CORRECTION_PREDICTIONS_PATH does not exist: {override_path}')
        return override_path.parent, {'canonical': override_path}
    latest_dir = workspace.runs_dir / 'ocr_training' / 'latest'
    canonical_candidates = [latest_dir / f'{source_name}_predictions.jsonl']
    if source_name == 'target_only':
        canonical_candidates.append(latest_dir / 'target_predictions.jsonl')
    for canonical_path in canonical_candidates:
        if canonical_path.exists():
            return canonical_path.parent, {'canonical': canonical_path}

    run_prefix = 'trocr_target_only'
    if source_name not in {'', 'target_only'}:
        run_prefix = source_name
    label_specific = sorted(workspace.runs_dir.glob(f'{run_prefix}_{run_label}_*'))
    candidates = label_specific or sorted(workspace.runs_dir.glob(f'{run_prefix}_*'))
    runs_with_predictions: list[tuple[str, Path, dict[str, Path]]] = []
    for run_dir in candidates:
        if not run_dir.is_dir():
            continue
        paths = {}
        for split in ('train', 'val', 'test'):
            path = run_dir / f'predictions_{split}.jsonl'
            if path.exists():
                paths[split] = path
        if paths:
            runs_with_predictions.append((run_dir.name, run_dir, paths))
    if not runs_with_predictions:
        return None, {}
    _, run_dir, paths = sorted(runs_with_predictions, key=lambda item: item[0])[-1]
    return run_dir, paths


def build_correction_pairs(*, correction_manifest_path: Path, prediction_paths: dict[str, Path], source_ocr_run_dir: Path | None) -> tuple[list[dict], dict]:
    correction_rows = read_jsonl(correction_manifest_path)
    correction_by_sample = {row.get('sample_id'): row for row in correction_rows}
    pairs: list[dict] = []
    skipped = []
    for split_name, prediction_path in sorted(prediction_paths.items()):
        for pred in read_jsonl(prediction_path):
            sample_id = pred.get('sample_id')
            gt = correction_by_sample.get(sample_id)
            if not gt:
                skipped.append({'sample_id': sample_id, 'reason': 'missing_correction_manifest_row', 'prediction_path': str(prediction_path)})
                continue
            raw_ocr_text = prediction_text_from_row(pred)
            ground_truth = normalize_text(gt.get('normalized_text') or gt.get('raw_text') or pred.get('reference_normalized') or '')
            if not raw_ocr_text or not ground_truth:
                skipped.append({'sample_id': sample_id, 'reason': 'empty_prediction_or_ground_truth', 'prediction_path': str(prediction_path)})
                continue
            dataset_split = pred.get('dataset_split') or gt.get('dataset_split') or split_name
            source_ocr_run_id = pred.get('source_ocr_run_id') or (source_ocr_run_dir.name if source_ocr_run_dir else prediction_path.parent.name)
            pairs.append({
                'pair_id': f"{source_ocr_run_id}_{sample_id}",
                'sample_id': sample_id,
                'page_id': gt.get('page_id') or pred.get('page_id'),
                'group_id': gt.get('group_id') or pred.get('group_id'),
                'dataset_split': dataset_split,
                'source_ocr_run_id': source_ocr_run_id,
                'source_predictions_path': str(prediction_path),
                'correction_manifest_path': str(correction_manifest_path),
                'raw_ocr_text': raw_ocr_text,
                'ground_truth': ground_truth,
                'raw_cer': cer(ground_truth, raw_ocr_text),
                'raw_wer': wer(ground_truth, raw_ocr_text),
                'warnings': [],
            })
    audit = {
        'correction_manifest_path': str(correction_manifest_path),
        'source_ocr_run_dir': str(source_ocr_run_dir) if source_ocr_run_dir else None,
        'prediction_paths': {split: str(path) for split, path in prediction_paths.items()},
        'correction_manifest_rows': len(correction_rows),
        'pair_count': len(pairs),
        'skipped_count': len(skipped),
        'skipped_examples': skipped[:20],
    }
    return pairs, audit


correction_manifest_path = Path(CORRECTION_MANIFEST_PATH) if CORRECTION_MANIFEST_PATH else WORKSPACE.manifests_dir / f'target_lines_{DATA_RUN_LABEL}.jsonl'
source_ocr_run_dir, prediction_paths = discover_prediction_paths_for_source(WORKSPACE, DATA_RUN_LABEL, CORRECTION_SOURCE_NAME)
correction_pairs, pair_audit = build_correction_pairs(
    correction_manifest_path=correction_manifest_path,
    prediction_paths=prediction_paths,
    source_ocr_run_dir=source_ocr_run_dir,
)

correction_pairs_path = WORKSPACE.manifests_dir / f'correction_pairs_{CORRECTION_SOURCE_NAME}_{DATA_RUN_LABEL}.jsonl'
write_jsonl(correction_pairs_path, correction_pairs)
write_json(CORRECTION_RUN_DIR / 'correction_pair_audit.json', pair_audit)
write_jsonl(CORRECTION_RUN_DIR / 'correction_pairs.jsonl', correction_pairs)

pairs_df = pd.DataFrame(correction_pairs)
summary_table(
    {
        'correction_manifest_path': str(correction_manifest_path),
        'source_ocr_run_dir': str(source_ocr_run_dir) if source_ocr_run_dir else None,
        'prediction_splits_found': ', '.join(sorted(prediction_paths)) or 'none',
        'correction_pairs_path': str(correction_pairs_path),
        'pair_count': len(correction_pairs),
        'skipped_count': pair_audit['skipped_count'],
    },
    'Correction pair discovery',
)
if pairs_df.empty:
    display(Markdown('No correction pairs were found. Check the OCR prediction path/source and correction manifest selection.'))
else:
    display(pairs_df.groupby('dataset_split').size().reset_index(name='pairs'))
    display(pairs_df.sort_values('raw_cer', ascending=False).head(8)[['dataset_split', 'sample_id', 'raw_cer', 'raw_wer', 'raw_ocr_text', 'ground_truth']])


## BART Checkpoint Resolution, Frozen Correction, and Canonical Training Controls

In [ ]:
# ## Seq2seq deps, BART model load, and training/eval entrypoints for the corrector.
# Why: seq2seq correction is the canonical post-OCR target-text stage, and this notebook now preserves full trainer history for later graphing.
def require_seq2seq_deps():
    import torch
    from transformers import (
        AutoModelForSeq2SeqLM,
        AutoTokenizer,
        DataCollatorForSeq2Seq,
        EarlyStoppingCallback,
        Seq2SeqTrainer,
        Seq2SeqTrainingArguments,
    )
    return torch, AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, EarlyStoppingCallback, Seq2SeqTrainer, Seq2SeqTrainingArguments


def _looks_like_bart_config(config_path: Path) -> bool:
    try:
        payload = json.loads(config_path.read_text(encoding='utf-8'))
    except Exception:
        return False
    return payload.get('model_type') == 'bart' or 'bart' in str(payload.get('_name_or_path', '')).lower()


def find_local_bart_checkpoint(workspace: Workspace, model_id: str) -> Path | None:
    safe_id = model_id.replace('/', '--')
    short_id = model_id.split('/')[-1]
    direct_candidates = [
        workspace.models_dir / model_id,
        workspace.models_dir / safe_id,
        workspace.models_dir / short_id,
        workspace.runs_dir / safe_id,
        workspace.runs_dir / short_id,
    ]
    for candidate in direct_candidates:
        config_path = candidate / 'config.json'
        if config_path.exists() and _looks_like_bart_config(config_path):
            return candidate
    cache_roots = []
    for env_name in ('HF_HOME', 'TRANSFORMERS_CACHE', 'HUGGINGFACE_HUB_CACHE'):
        value = os.environ.get(env_name)
        if value:
            cache_roots.append(Path(value))
    cache_roots.extend([
        Path.home() / '.cache' / 'huggingface' / 'hub',
        Path(os.environ.get('USERPROFILE', str(Path.home()))) / '.cache' / 'huggingface' / 'hub',
    ])
    for root in cache_roots:
        snapshots = root / f'models--{safe_id}' / 'snapshots'
        if not snapshots.exists():
            continue
        for snapshot in sorted([item for item in snapshots.iterdir() if item.is_dir()], reverse=True):
            config_path = snapshot / 'config.json'
            if config_path.exists() and _looks_like_bart_config(config_path):
                return snapshot
    return None


def load_seq2seq_with_preferred_fallback(workspace: Workspace, *, device: str | None = None) -> tuple[dict, object, object]:
    torch, AutoTokenizer, AutoModelForSeq2SeqLM, *_ = require_seq2seq_deps()
    resolved_device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    if resolved_device.startswith('cuda'):
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.backends.cudnn.benchmark = True
    attempts = []
    local_historical = find_local_bart_checkpoint(workspace, HISTORICAL_BART_MODEL)
    load_candidates = []
    if BART_BASE_CHECKPOINT_OVERRIDE:
        load_candidates.append(('explicit_bart_base_override', BART_BASE_CHECKPOINT_OVERRIDE, BART_BASE_CHECKPOINT_OVERRIDE))
    if local_historical:
        load_candidates.append(('local_historical_bart', HISTORICAL_BART_MODEL, str(local_historical)))
    load_candidates.append(('hf_historical_bart', HISTORICAL_BART_MODEL, HISTORICAL_BART_MODEL))
    load_candidates.append(('hf_fallback_bart', FALLBACK_BART_MODEL, FALLBACK_BART_MODEL))
    for resolution, base_checkpoint_used, load_path in load_candidates:
        try:
            tokenizer = AutoTokenizer.from_pretrained(load_path)
            model = AutoModelForSeq2SeqLM.from_pretrained(load_path).to(resolved_device)
            model_info = {
                'base_checkpoint_used': base_checkpoint_used,
                'load_path': load_path,
                'resolution': resolution,
                'device': resolved_device,
                'fallback_attempts': attempts,
            }
            return model_info, tokenizer, model
        except Exception as exc:
            attempts.append({'resolution': resolution, 'base_checkpoint': base_checkpoint_used, 'load_path': str(load_path), 'error': repr(exc)})
    raise RuntimeError(f'Could not load historical or fallback BART checkpoints. Attempts: {attempts}')


def select_pairs(rows: list[dict], split: str | None = None, max_samples: int | None = None) -> list[dict]:
    selected = [row for row in rows if split is None or row.get('dataset_split') == split]
    return selected[:max_samples] if max_samples is not None else selected


def correction_summary(rows: list[dict], *, label: str) -> dict:
    if not rows:
        return {'label': label, 'sample_count': 0}
    return {
        'label': label,
        'sample_count': len(rows),
        'mean_raw_cer': float(np.mean([row['raw_cer'] for row in rows])),
        'mean_corrected_cer': float(np.mean([row['corrected_cer'] for row in rows])),
        'mean_raw_wer': float(np.mean([row['raw_wer'] for row in rows])),
        'mean_corrected_wer': float(np.mean([row['corrected_wer'] for row in rows])),
        'cer_delta_raw_minus_corrected': float(np.mean([row['raw_cer'] for row in rows]) - np.mean([row['corrected_cer'] for row in rows])),
        'wer_delta_raw_minus_corrected': float(np.mean([row['raw_wer'] for row in rows]) - np.mean([row['corrected_wer'] for row in rows])),
    }


def generate_bart_corrections(*, pairs: list[dict], model_info: dict, tokenizer, model, correction_mode: str, correction_run_id: str, fine_tune_dataset_path: Path, fine_tuned_checkpoint_path: Path | None, batch_size: int, max_samples: int | None = None) -> list[dict]:
    torch, *_ = require_seq2seq_deps()
    selected = pairs[:max_samples] if max_samples is not None else pairs
    if not selected:
        return []
    model.eval()
    device = model_info['device']
    rows: list[dict] = []
    model.eval()
    with torch.inference_mode():
        for start in range(0, len(selected), batch_size):
            batch = selected[start:start + batch_size]
            encoded = tokenizer(
                [row['raw_ocr_text'] for row in batch],
                return_tensors='pt',
                padding=True,
                truncation=True,
                max_length=MAX_SOURCE_LENGTH,
            ).to(device)
            generated = model.generate(**encoded, max_length=MAX_TARGET_LENGTH, num_beams=GENERATION_NUM_BEAMS)
            decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
            for row, corrected_text in zip(batch, decoded):
                corrected_text = normalize_text(corrected_text)
                ground_truth = row.get('ground_truth', '')
                rows.append({
                    'correction_run_id': correction_run_id,
                    'correction_mode': correction_mode,
                    'base_checkpoint_used': model_info['base_checkpoint_used'],
                    'base_checkpoint_resolution': model_info['resolution'],
                    'base_checkpoint_load_path': model_info['load_path'],
                    'fine_tune_dataset_path': str(fine_tune_dataset_path),
                    'fine_tuned_checkpoint_path': str(fine_tuned_checkpoint_path) if fine_tuned_checkpoint_path else None,
                    'source_ocr_run_id': row.get('source_ocr_run_id'),
                    'source_predictions_path': row.get('source_predictions_path'),
                    'sample_id': row.get('sample_id'),
                    'page_id': row.get('page_id'),
                    'group_id': row.get('group_id'),
                    'dataset_split': row.get('dataset_split'),
                    'raw_ocr_text': row.get('raw_ocr_text', ''),
                    'corrected_text': corrected_text,
                    'ground_truth': ground_truth,
                    'raw_cer': row.get('raw_cer'),
                    'raw_wer': row.get('raw_wer'),
                    'corrected_cer': cer(ground_truth, corrected_text),
                    'corrected_wer': wer(ground_truth, corrected_text),
                    'warnings': row.get('warnings', []),
                })
    return rows


all_corrected_rows: list[dict] = []
frozen_summary = {'label': 'frozen_bart', 'sample_count': 0, 'status': 'not_run'}
base_model_info = {'base_checkpoint_used': HISTORICAL_BART_MODEL, 'resolution': 'not_loaded', 'load_path': None, 'fallback_attempts': []}

if RUN_FROZEN_BART and correction_pairs:
    test_pairs = select_pairs(correction_pairs, 'test', MAX_FROZEN_TEST_SAMPLES)
    base_model_info, frozen_tokenizer, frozen_model = load_seq2seq_with_preferred_fallback(WORKSPACE)
    frozen_rows = generate_bart_corrections(
        pairs=test_pairs,
        model_info=base_model_info,
        tokenizer=frozen_tokenizer,
        model=frozen_model,
        correction_mode='frozen_bart',
        correction_run_id=CORRECTION_RUN_ID,
        fine_tune_dataset_path=correction_pairs_path,
        fine_tuned_checkpoint_path=None,
        batch_size=BART_BATCH_SIZE,
    )
    frozen_path = CORRECTION_RUN_DIR / 'corrected_predictions_frozen_bart.jsonl'
    write_jsonl(frozen_path, frozen_rows)
    frozen_summary = correction_summary(frozen_rows, label='frozen_bart')
    frozen_summary.update({'status': 'completed', 'output_path': str(frozen_path), **base_model_info})
    write_json(frozen_path.with_suffix('.summary.json'), frozen_summary)
    all_corrected_rows.extend(frozen_rows)
    summary_table(frozen_summary, 'Frozen BART correction summary')
elif not correction_pairs:
    display(Markdown('Frozen BART skipped because no correction pairs are available.'))
else:
    display(Markdown('Frozen BART is ready. Set `RUN_FROZEN_BART=1` to run correction.'))


## Fine-Tune BART On Target OCR Correction Pairs With Full Training History

In [ ]:
# ## torch Dataset: tokenize source OCR and target gold text to seq2seq batch tensors.
class CorrectionTextDataset:
    def __init__(self, pairs: list[dict], tokenizer, *, max_source_length: int, max_target_length: int):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_source_length = max_source_length
        self.max_target_length = max_target_length

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, idx: int) -> dict:
        row = self.pairs[idx]
        model_inputs = self.tokenizer(row['raw_ocr_text'], max_length=self.max_source_length, truncation=True)
        labels = self.tokenizer(text_target=row['ground_truth'], max_length=self.max_target_length, truncation=True)
        model_inputs['labels'] = labels['input_ids']
        return model_inputs


def _seq2seq_args(output_dir: Path, Seq2SeqTrainingArguments, torch) -> object:
    params = inspect.signature(Seq2SeqTrainingArguments.__init__).parameters
    kwargs = {
        'output_dir': str(output_dir),
        'num_train_epochs': BART_EPOCHS,
        'per_device_train_batch_size': BART_BATCH_SIZE,
        'per_device_eval_batch_size': BART_BATCH_SIZE,
        'gradient_accumulation_steps': BART_GRADIENT_ACCUMULATION,
        'precision': BART_PRECISION,
        'dataloader_num_workers': BART_DATALOADER_WORKERS,
        'learning_rate': BART_LEARNING_RATE,
        'warmup_ratio': BART_WARMUP_RATIO,
        'lr_scheduler_type': BART_LR_SCHEDULER,
        'save_strategy': 'epoch',
        'logging_steps': 25,
        'save_total_limit': BART_SAVE_TOTAL_LIMIT,
        'load_best_model_at_end': True,
        'metric_for_best_model': 'cer',
        'greater_is_better': False,
        'predict_with_generate': True,
        'generation_max_length': MAX_TARGET_LENGTH,
        'report_to': [],
        'dataloader_num_workers': BART_DATALOADER_WORKERS,
        'dataloader_pin_memory': bool(torch.cuda.is_available()),
        'bf16': bool(torch.cuda.is_available() and (BART_PRECISION == 'bf16' or (BART_PRECISION == 'auto' and torch.cuda.is_bf16_supported()))),
        'fp16': bool(torch.cuda.is_available() and BART_PRECISION == 'fp16'),
        'tf32': bool(torch.cuda.is_available()),
        'seed': BART_SEED,
        'data_seed': BART_SEED,
    }
    if 'eval_strategy' in params:
        kwargs['eval_strategy'] = 'epoch'
    else:
        kwargs['evaluation_strategy'] = 'epoch'
    if 'generation_num_beams' in params:
        kwargs['generation_num_beams'] = GENERATION_NUM_BEAMS
    return Seq2SeqTrainingArguments(**{key: value for key, value in kwargs.items() if key in params})


def train_bart_correction(*, train_pairs: list[dict], val_pairs: list[dict], output_dir: Path) -> tuple[dict, object, object, Path]:
    torch, AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, EarlyStoppingCallback, Seq2SeqTrainer, Seq2SeqTrainingArguments = require_seq2seq_deps()
    if not train_pairs or not val_pairs:
        raise ValueError(f'Need train and val correction pairs. train={len(train_pairs)} val={len(val_pairs)}')
    model_info, tokenizer, model = load_seq2seq_with_preferred_fallback(WORKSPACE)
    train_dataset = CorrectionTextDataset(train_pairs, tokenizer, max_source_length=MAX_SOURCE_LENGTH, max_target_length=MAX_TARGET_LENGTH)
    val_dataset = CorrectionTextDataset(val_pairs, tokenizer, max_source_length=MAX_SOURCE_LENGTH, max_target_length=MAX_TARGET_LENGTH)
    collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        if isinstance(predictions, tuple):
            predictions = predictions[0]
        predictions = np.asarray(predictions)
        if predictions.ndim == 3:
            predictions = predictions.argmax(axis=-1)
        vocab_size = len(tokenizer)
        fallback_id = tokenizer.unk_token_id if tokenizer.unk_token_id is not None else tokenizer.pad_token_id
        predictions = np.where(predictions < 0, tokenizer.pad_token_id, predictions)
        predictions = np.where(predictions >= vocab_size, fallback_id, predictions)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        labels = np.where(labels < 0, tokenizer.pad_token_id, labels)
        labels = np.where(labels >= vocab_size, fallback_id, labels)
        decoded_preds = [normalize_text(item) for item in tokenizer.batch_decode(predictions.astype(int).tolist(), skip_special_tokens=True)]
        decoded_labels = [normalize_text(item) for item in tokenizer.batch_decode(labels.astype(int).tolist(), skip_special_tokens=True)]
        return {
            'cer': float(np.mean([cer(ref, hyp) for ref, hyp in zip(decoded_labels, decoded_preds)])),
            'wer': float(np.mean([wer(ref, hyp) for ref, hyp in zip(decoded_labels, decoded_preds)])),
        }

    training_output_dir = ensure_dir(output_dir / 'trainer_work')
    args = _seq2seq_args(training_output_dir, Seq2SeqTrainingArguments, torch)
    trainer_kwargs = {
        'model': model,
        'args': args,
        'train_dataset': train_dataset,
        'eval_dataset': val_dataset,
        'data_collator': collator,
        'compute_metrics': compute_metrics,
    }
    if not BART_DISABLE_EARLY_STOPPING:
        trainer_kwargs['callbacks'] = [EarlyStoppingCallback(early_stopping_patience=BART_EARLY_STOPPING_PATIENCE, early_stopping_threshold=BART_EARLY_STOPPING_THRESHOLD)]
    trainer_params = inspect.signature(Seq2SeqTrainer.__init__).parameters
    if 'tokenizer' in trainer_params:
        trainer_kwargs['tokenizer'] = tokenizer
    elif 'processing_class' in trainer_params:
        trainer_kwargs['processing_class'] = tokenizer
    trainer = Seq2SeqTrainer(**trainer_kwargs)
    train_result = trainer.train()
    trainer.save_state()
    checkpoint_dir = ensure_dir(output_dir / 'bart_finetuned_checkpoint')
    trainer.save_model(str(checkpoint_dir))
    tokenizer.save_pretrained(str(checkpoint_dir))
    trainer_state_path = output_dir / 'trainer_state_full.json'
    trainer.state.save_to_json(str(trainer_state_path))
    log_history_path = output_dir / 'trainer_log_history.csv'
    write_csv(log_history_path, trainer.state.log_history)
    write_json(output_dir / 'training_args.json', trainer.args.to_dict())
    train_summary = {
        'status': 'completed',
        'train_samples': len(train_pairs),
        'val_samples': len(val_pairs),
        'epochs': BART_EPOCHS,
        'learning_rate': BART_LEARNING_RATE,
        'warmup_ratio': BART_WARMUP_RATIO,
        'lr_scheduler_type': BART_LR_SCHEDULER,
        'batch_size': BART_BATCH_SIZE,
        'gradient_accumulation_steps': BART_GRADIENT_ACCUMULATION,
        'precision': BART_PRECISION,
        'dataloader_num_workers': BART_DATALOADER_WORKERS,
        'train_runtime': getattr(train_result, 'metrics', {}).get('train_runtime'),
        'train_loss': getattr(train_result, 'metrics', {}).get('train_loss'),
        'fine_tuned_checkpoint_path': str(checkpoint_dir),
        'trainer_state_path': str(trainer_state_path),
        'trainer_log_history_csv': str(log_history_path),
        'best_global_step': trainer.state.best_global_step,
        'best_metric': trainer.state.best_metric,
        'best_model_checkpoint': trainer.state.best_model_checkpoint,
        'global_step': trainer.state.global_step,
        'num_train_epochs_configured': trainer.state.num_train_epochs,
        'max_steps': trainer.state.max_steps,
        'seed': BART_SEED,
        'save_total_limit': BART_SAVE_TOTAL_LIMIT,
        'early_stopping_disabled': BART_DISABLE_EARLY_STOPPING,
        'early_stopping_patience': None if BART_DISABLE_EARLY_STOPPING else BART_EARLY_STOPPING_PATIENCE,
        'early_stopping_threshold': None if BART_DISABLE_EARLY_STOPPING else BART_EARLY_STOPPING_THRESHOLD,
        **model_info,
    }
    write_json(output_dir / 'bart_finetune_summary.json', train_summary)
    return train_summary, tokenizer, trainer.model, checkpoint_dir


finetuned_summary = {'label': 'finetuned_bart', 'sample_count': 0, 'status': 'not_run'}
finetuned_checkpoint_path = None
if RUN_BART_FINETUNE and correction_pairs:
    train_pairs = select_pairs(correction_pairs, 'train', MAX_FINETUNE_TRAIN_SAMPLES)
    val_pairs = select_pairs(correction_pairs, 'val', MAX_FINETUNE_VAL_SAMPLES)
    test_pairs = select_pairs(correction_pairs, 'test', MAX_FINETUNE_TEST_SAMPLES)
    if not train_pairs or not val_pairs:
        display(Markdown(f'BART fine-tune skipped: train pairs={len(train_pairs)}, val pairs={len(val_pairs)}. Run Part 3 predictions for train and val first.'))
    else:
        finetuned_summary, tuned_tokenizer, tuned_model, finetuned_checkpoint_path = train_bart_correction(train_pairs=train_pairs, val_pairs=val_pairs, output_dir=CORRECTION_RUN_DIR)
        tuned_model_info = {
            'base_checkpoint_used': finetuned_summary['base_checkpoint_used'],
            'load_path': str(finetuned_checkpoint_path),
            'resolution': 'finetuned_bart_checkpoint',
            'device': finetuned_summary['device'],
            'fallback_attempts': finetuned_summary.get('fallback_attempts', []),
        }
        tuned_rows = generate_bart_corrections(
            pairs=test_pairs,
            model_info=tuned_model_info,
            tokenizer=tuned_tokenizer,
            model=tuned_model,
            correction_mode='finetuned_bart',
            correction_run_id=CORRECTION_RUN_ID,
            fine_tune_dataset_path=correction_pairs_path,
            fine_tuned_checkpoint_path=finetuned_checkpoint_path,
            batch_size=BART_BATCH_SIZE,
        )
        tuned_path = CORRECTION_RUN_DIR / 'corrected_predictions_finetuned_bart.jsonl'
        write_jsonl(tuned_path, tuned_rows)
        tuned_eval_summary = correction_summary(tuned_rows, label='finetuned_bart')
        tuned_eval_summary.update({'status': 'completed', 'output_path': str(tuned_path), 'fine_tuned_checkpoint_path': str(finetuned_checkpoint_path)})
        write_json(tuned_path.with_suffix('.summary.json'), tuned_eval_summary)
        finetuned_summary.update(tuned_eval_summary)
        all_corrected_rows.extend(tuned_rows)
        summary_table(finetuned_summary, 'Fine-tuned BART correction summary')
elif not correction_pairs:
    display(Markdown('BART fine-tune skipped because no correction pairs are available.'))
else:
    display(Markdown('BART fine-tuning is ready. Set `RUN_BART_FINETUNE=1` after train/val/test predictions exist.'))

## Final Raw OCR vs BART-Corrected OCR Evaluation

In [ ]:
# ## Scoring helpers: CER/WER on correction outputs; comparison tables vs raw TrOCR.
# Why: report both CER and WER to capture character fidelity and lexical readability.
RAW_SYSTEM_LABEL_OVERRIDE = os.environ.get('RAW_SYSTEM_LABEL_OVERRIDE', '').strip()
RAW_SYSTEM_LABEL = RAW_SYSTEM_LABEL_OVERRIDE or ({
    'target_only': 'raw_scratch_target_trocr',
    'tesseract_gle_lat_target': 'raw_tesseract_gle_lat',
}.get(CORRECTION_SOURCE_NAME, f"raw_{CORRECTION_SOURCE_NAME}".strip('_')) or 'raw_ocr')

def raw_summary_from_pairs(pairs: list[dict], *, split: str = 'test', max_samples: int | None = None, label: str = RAW_SYSTEM_LABEL) -> dict:
    selected = select_pairs(pairs, split, max_samples)
    if not selected:
        return {'label': label, 'sample_count': 0}
    return {
        'label': label,
        'sample_count': len(selected),
        'mean_cer': float(np.mean([row['raw_cer'] for row in selected])),
        'mean_wer': float(np.mean([row['raw_wer'] for row in selected])),
    }


def comparison_rows_from_summaries(raw_summary: dict, correction_summaries: list[dict]) -> list[dict]:
    rows = []
    if raw_summary.get('sample_count', 0):
        rows.append({
            'system': raw_summary.get('label', RAW_SYSTEM_LABEL),
            'sample_count': raw_summary['sample_count'],
            'mean_cer': raw_summary['mean_cer'],
            'mean_wer': raw_summary['mean_wer'],
            'cer_delta_vs_raw': 0.0,
            'wer_delta_vs_raw': 0.0,
        })
    raw_cer = raw_summary.get('mean_cer')
    raw_wer = raw_summary.get('mean_wer')
    for summary in correction_summaries:
        if not summary.get('sample_count', 0) or 'mean_corrected_cer' not in summary:
            continue
        rows.append({
            'system': summary['label'],
            'sample_count': summary['sample_count'],
            'mean_cer': summary['mean_corrected_cer'],
            'mean_wer': summary['mean_corrected_wer'],
            'cer_delta_vs_raw': (raw_cer - summary['mean_corrected_cer']) if raw_cer is not None else None,
            'wer_delta_vs_raw': (raw_wer - summary['mean_corrected_wer']) if raw_wer is not None else None,
        })
    return rows


raw_test_summary = raw_summary_from_pairs(correction_pairs, split='test', max_samples=None, label=RAW_SYSTEM_LABEL)
comparison_rows = comparison_rows_from_summaries(raw_test_summary, [frozen_summary, finetuned_summary])
comparison_df = pd.DataFrame(comparison_rows)
comparison_path = CORRECTION_RUN_DIR / 'final_correction_comparison.csv'
if not comparison_df.empty:
    comparison_df.to_csv(comparison_path, index=False)
    display(comparison_df)
else:
    display(Markdown('No correction comparison rows yet. Run frozen BART and/or fine-tuned BART correction.'))

summary_payload = {
    'correction_run_id': CORRECTION_RUN_ID,
    'data_run_label': DATA_RUN_LABEL,
    'source_ocr_run_dir': str(source_ocr_run_dir) if source_ocr_run_dir else None,
    'correction_pairs_path': str(correction_pairs_path),
    'corrected_predictions_frozen_bart_path': frozen_summary.get('output_path'),
    'corrected_predictions_finetuned_bart_path': finetuned_summary.get('output_path'),
    'final_correction_comparison_csv': str(comparison_path),
    'base_checkpoint_preference': HISTORICAL_BART_MODEL,
    'fallback_checkpoint': FALLBACK_BART_MODEL,
    'irish_bert_note': 'Irish BERT checkpoints are encoder/fill-mask models and are reserved for optional reranking/scoring, not direct OCR correction generation.',
    'base_model_info': base_model_info,
    'raw_test_summary': raw_test_summary,
    'frozen_bart_summary': frozen_summary,
    'finetuned_bart_summary': finetuned_summary,
    'pair_audit': pair_audit,
}
write_json(CORRECTION_RUN_DIR / 'overall_summary.json', summary_payload)

if not comparison_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    ordered = comparison_df.sort_values('mean_cer')
    axes[0].bar(ordered['system'], ordered['mean_cer'])
    axes[0].set_title('Correction CER')
    axes[0].set_ylabel('Mean CER, lower is better')
    axes[0].tick_params(axis='x', rotation=20)
    axes[1].bar(ordered['system'], ordered['mean_wer'])
    axes[1].set_title('Correction WER')
    axes[1].set_ylabel('Mean WER, lower is better')
    axes[1].tick_params(axis='x', rotation=20)
    plot_path = CORRECTION_RUN_DIR / 'correction_metrics.png'
    fig.savefig(plot_path, dpi=180)
    display(fig)
    plt.close(fig)
    display(Markdown(f'Correction comparison written to `{comparison_path}`. Plot written to `{plot_path}`.'))

display(Markdown(f"Frozen BART predictions: `{summary_payload['corrected_predictions_frozen_bart_path']}`"))
display(Markdown(f"Fine-tuned BART predictions: `{summary_payload['corrected_predictions_finetuned_bart_path']}`"))
